# Proyecto GAN — Síntesis de Cuerpo Humano

**Autores:** Santiago Álvarez Geanta, Alejandro García Belmonte, Guillermo García Blázquez, Taron Sargsyan  
**Asignatura:** Redes Neuronales · Universidad de Alicante

---

## Índice

1. Introducción — ¿qué es una GAN y por qué la usamos para cuerpos humanos?
2. Dataset — ¿qué es AMASS y qué contienen los datos?
3. Preprocesado — de poses a joints normalizados
4. GAN Tabular — generando coordenadas 3D de articulaciones
5. GAN Imagen — generando siluetas de cuerpo humano
6. GAN 3D — generando nubes de puntos (mallas)
7. Métricas — cómo medimos si el resultado es bueno
8. Conclusiones y próximos pasos

> Este notebook está pensado para que alguien **sin conocimientos de programación** pueda seguirlo. Cada sección explica primero **qué** estamos haciendo y **por qué**, y después muestra el código que lo ejecuta.

---
## 0. Preparación del entorno

Antes de nada, inicializamos las rutas del proyecto. El módulo `Paths` crea los directorios necesarios (`internal/data/`, `internal/experiments/`, etc.) si no existen. Así todas las partes del proyecto saben dónde guardar y leer datos.

In [ ]:
# Cambiamos al directorio raíz del proyecto para que los imports funcionen.
import os
os.chdir('..')

# Inicializa la estructura de carpetas del proyecto.
from src.config.paths import Paths
Paths.init_project()

print(f'Raíz del proyecto: {Paths.ROOT}')
print(f'Datos generados se guardarán en: {Paths.DATA_DIR}')
print(f'Checkpoints de entrenamiento en: {Paths.EXPERIMENTS_DIR}')

---
## 1. Introducción — ¿qué es una GAN?

Una **GAN** (*Generative Adversarial Network*, red generativa antagónica) es una pareja de redes neuronales que aprenden **compitiendo entre sí**:

- El **Generador (G)** parte de ruido aleatorio e intenta crear datos que parezcan reales (por ejemplo, la pose de un cuerpo humano).
- El **Discriminador (D)** recibe datos reales y generados, e intenta distinguirlos.

El juego es: G quiere engañar a D, y D quiere no dejarse engañar. Tras muchas iteraciones, G aprende a generar ejemplos tan realistas que D ya no puede distinguirlos. En ese momento decimos que la GAN **ha aprendido la distribución de los datos**.

### ¿Por qué cuerpos humanos?

Generar cuerpos humanos plausibles tiene aplicaciones en realidad virtual, videojuegos, antropometría clínica, moda, entrenamiento de robots, etc. Es un problema **no trivial** porque un cuerpo no es cualquier punto en el espacio: hay proporciones, articulaciones y restricciones biomecánicas que deben respetarse.

### Tres modalidades

En este proyecto entrenamos **tres GANs distintas**, cada una trabaja con una representación diferente del cuerpo:

| Modalidad | Qué genera | Dimensión |
|-----------|-----------|-----------|
| **Tabular** | Coordenadas 3D de 24 articulaciones | Vector de 72 números |
| **Imagen** | Silueta del cuerpo en 2D | Imagen 128×128 RGB |
| **3D** | Nube de puntos con la forma del cuerpo | 6890 puntos × 3 coords |

Todas usan la variante **WGAN-GP** (Wasserstein GAN con Gradient Penalty), que es más estable de entrenar que la GAN clásica.

---
## 2. Dataset — AMASS, TNT15, NOMO3D

Usamos **tres fuentes de datos distintas**, una para cada modalidad:

### 2.1 AMASS — para la GAN Tabular

[AMASS](https://amass.is.tue.mpg.de/) es un meta-dataset de *motion capture* que unifica decenas de datasets bajo un formato común: el modelo **SMPL**.

- **SMPL** es un modelo matemático del cuerpo humano que, dados unos parámetros de forma (`betas`) y de pose (`poses`), produce una malla 3D de 6890 vértices.
- De cada secuencia de AMASS extraemos frames con las **coordenadas 3D de 24 articulaciones** (pelvis, rodillas, codos, muñecas, cabeza, etc.).

Los archivos de AMASS vienen en formato `.npz` y contienen, entre otras cosas:
- `poses`: una matriz `(N, 156)` donde cada fila es la pose de un frame.
- `betas`: un vector de 10 números que describen la forma del cuerpo.
- `gender`: `male`, `female` o `neutral`.

### 2.2 TNT15 — para la GAN Imagen

[TNT15](https://www.tnt.uni-hannover.de/) contiene imágenes segmentadas de personas en distintas poses. Usamos la carpeta `Images/mr/` (~25k PNGs), donde cada imagen es una **silueta blanca sobre fondo negro**.

### 2.3 NOMO3D — para la GAN 3D

**NOMO3D** es una colección de escaneos 3D de personas reales en malla OBJ. Cada escaneo tiene ~57.000 vértices. Los submuestreamos a **6890 puntos** para que coincidan con la topología SMPL.

In [ ]:
# Listamos los datasets disponibles en disco.
from pathlib import Path

print('📂 AMASS (.npz a añadir por el usuario):')
npz_files = list(Paths.AMASS_DIR.glob('**/*.npz'))
print(f'   {len(npz_files)} archivos .npz encontrados')

print('\n📂 TNT15 (imágenes):')
tnt15_pngs = list(Paths.TNT15_IMAGES_DIR.glob('**/*_segmented.png')) if Paths.TNT15_IMAGES_DIR.exists() else []
print(f'   {len(tnt15_pngs)} imágenes PNG segmentadas')

print('\n📂 NOMO3D (escaneos 3D):')
nomo_female = list(Paths.NOMO3D_FEMALE_DIR.glob('*.obj')) if Paths.NOMO3D_FEMALE_DIR.exists() else []
nomo_male = list(Paths.NOMO3D_MALE_DIR.glob('*.obj')) if Paths.NOMO3D_MALE_DIR.exists() else []
print(f'   {len(nomo_female)} female + {len(nomo_male)} male = {len(nomo_female)+len(nomo_male)} OBJs')

---
## 3. Preprocesado — de poses SMPL a joints normalizados

Los archivos de AMASS no contienen directamente las coordenadas 3D de las articulaciones: contienen **parámetros** del modelo SMPL. Para obtener las coordenadas debemos hacer un *forward pass* del modelo (darle las poses y betas, y pedirle que devuelva los joints).

### Normalización

Un cuerpo humano puede estar en cualquier posición del espacio y tener cualquier tamaño. Si dejamos eso así, la GAN tendría que aprender también la traslación y la escala, lo cual es ruido para nuestro objetivo. Por eso normalizamos:

1. **Centrado en pelvis**: restamos la posición de la pelvis (joint 0) a todos los demás. La pelvis queda en el origen.
2. **Escala por altura**: dividimos por la distancia pelvis → cabeza. Todos los cuerpos tienen "altura 1".
3. **Aplanamos** los 24 joints × 3 coordenadas = **72 números** por cuerpo.

Cada cuerpo se convierte así en un **vector de 72 dimensiones**.

> **Nota**: este paso necesita que haya archivos `.npz` de AMASS en `src/dataset/AMASS/`. Si no los hay, esta celda fallará — es responsabilidad del usuario descargarlos.

In [ ]:
# Pipeline completo de preprocesado AMASS → joints.npz
# Solo ejecutar si hay archivos .npz de AMASS disponibles.
from src.data.preprocess_joints import preprocess_and_save

try:
    joints = preprocess_and_save()
    print(f'\n✅ Resultado final: {joints.shape} — {joints.shape[0]} cuerpos de 72 dimensiones cada uno')
except FileNotFoundError as e:
    print(f'⚠️ {e}')
    print('Descarga AMASS desde https://amass.is.tue.mpg.de/ y coloca los .npz en src/dataset/AMASS/')

In [ ]:
# Visualización: un cuerpo real de AMASS
import numpy as np
import matplotlib.pyplot as plt

if Paths.JOINTS_NPZ.exists():
    data = np.load(str(Paths.JOINTS_NPZ))
    joints_flat = data['joints']
    
    # Cogemos un cuerpo al azar y lo mostramos como scatter 3D
    sample = joints_flat[np.random.randint(len(joints_flat))].reshape(24, 3)
    
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], s=40, c='navy')
    ax.set_title('Un cuerpo real de AMASS (24 joints normalizados)')
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    plt.show()
else:
    print('⚠️ joints.npz aún no existe. Ejecuta antes la celda de preprocesado.')

---
## 4. GAN Tabular — generando coordenadas 3D de joints

Esta es la GAN más sencilla y la **pieza central** del proyecto: genera directamente los 72 números que describen un cuerpo.

### Arquitectura

```
Generador:  ruido z (128) → MLP → 72 números
Discriminador:  72 números → MLP → un score (¿real o falso?)
```

Ambas redes son **MLP** (Multi-Layer Perceptron): capas totalmente conectadas con activaciones no lineales. El Generador acaba en `Tanh` para que los valores estén en el rango `[-1, 1]`. El Discriminador **no tiene activación final** porque usamos la formulación **Wasserstein**, donde la salida es una "puntuación" ilimitada en vez de una probabilidad.

### ¿Por qué WGAN-GP y no GAN clásica?

La GAN clásica (Goodfellow 2014) es famosa por ser **inestable**: a menudo el Generador se queda atascado o colapsa (*mode collapse*, genera siempre lo mismo). La variante **WGAN-GP** reemplaza la función de pérdida por una distancia de Wasserstein y añade una penalización sobre el gradiente del Discriminador para forzar que sea 1-Lipschitz. En la práctica, entrena de forma mucho más suave.

### Entrenamiento

- **Batch size**: 64
- **Learning rate**: 1e-4 (Adam β1=0.0, β2=0.9)
- **Épocas**: 200
- **Ratio D/G**: por cada paso del generador, hacemos 5 del discriminador. Así el D siempre va por delante del G y le da una señal útil.

> El entrenamiento completo tarda varias horas en GPU. Las siguientes celdas arrancan el proceso.

In [ ]:
# Inspeccionar la arquitectura de la Tabular GAN
from src.models.tabular_gan import Generator, Discriminator

G = Generator()
D = Discriminator()

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('=== Generador ===')
print(G)
print(f'Parámetros: {count_params(G):,}\n')

print('=== Discriminador ===')
print(D)
print(f'Parámetros: {count_params(D):,}')

In [ ]:
# Entrenamiento del Tabular GAN (~varias horas en GPU)
# Descomenta para ejecutar:
# from src.training.train_tabular import train_tabular
# train_tabular()

print('El entrenamiento se lanza con: python main.py --train-tabular')
print('o descomentando las líneas de arriba. Tarda varias horas en GPU.')

In [ ]:
# Visualizar la evolución de las losses tras el entrenamiento
import json

log_path = Paths.TABULAR_EXP_DIR / 'training_log.json'
if log_path.exists():
    with open(log_path) as f:
        log = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(log['d_loss'], label='D loss')
    axes[0].plot(log['g_loss'], label='G loss')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].set_title('Losses del Tabular GAN')
    
    axes[1].plot(log['gp'], color='red')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Gradient Penalty')
    axes[1].set_title('Gradient Penalty')
    plt.tight_layout(); plt.show()
else:
    print('Aún no hay logs — entrena primero el Tabular GAN.')

### Filtrado con el discriminador

Una vez entrenada la GAN, el Discriminador nos sirve también como **filtro de calidad**. La idea:

1. Generamos muchos cuerpos con el Generador (p. ej. 10.000).
2. Cada cuerpo lo pasamos por el Discriminador y aplicamos `sigmoid` para obtener un score en `[0, 1]`.
3. **Solo conservamos los que tengan score > 0.8** — los más realistas según D.

Estos "cuerpos perfectos" se guardan en `internal/data/generated_joints.npz` y alimentan las siguientes fases (renderizado de esqueletos e Image GAN).

In [ ]:
# Ejecutar el filtro — necesita un checkpoint entrenado del Tabular GAN
# from src.data.discriminator_filter import filter_generated_bodies
# filter_generated_bodies(n_target=1000, threshold=0.8)

print('Ejecutable vía: python main.py --filter --n 1000')

---
## 5. GAN Imagen — siluetas de cuerpo humano

Aquí generamos **imágenes 128×128 RGB** en lugar de vectores. Para imágenes la arquitectura óptima son las **convoluciones**, que aprovechan la estructura espacial (un píxel está relacionado con sus vecinos).

### Arquitectura: DCGAN

Usamos la arquitectura **DCGAN** (Radford et al. 2015), el referente clásico para GANs de imagen:

- **Generador**: parte de un vector de ruido → FC a 4×4×512 → 5 capas de **ConvTranspose2d** (que doblan el tamaño espacial y reducen canales) → imagen de 128×128×3.
- **Discriminador**: imagen 128×128×3 → 5 capas de **Conv2d** con stride 2 (que reducen a la mitad) → un score escalar.

### Datos de entrenamiento

Combinamos dos fuentes:
1. **TNT15**: ~25.000 PNGs segmentados de personas reales. Silueta clara sobre fondo negro.
2. **Renders propios**: convertimos los joints filtrados en imágenes de "muñeco de palitos" dibujando los huesos con Pillow.

El hecho de mezclar siluetas reales y esqueletos sintéticos ayuda a que la GAN vea **mayor variedad de poses**.

In [ ]:
# Visualizar un batch del dataset TNT15
from src.data.tn15_loader import TNT15Dataset
import torch

try:
    ds = TNT15Dataset()
    
    # Cogemos 8 imágenes aleatorias
    indices = np.random.choice(len(ds), 8, replace=False)
    imgs = torch.stack([ds[i] for i in indices])
    
    # Desnormalizamos [-1,1] → [0,1] para visualizar
    imgs = (imgs + 1) / 2
    
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, img in zip(axes.flat, imgs):
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.axis('off')
    plt.suptitle('Muestras reales de TNT15 (siluetas segmentadas)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'⚠️ {e}')

In [ ]:
# Renderizar esqueletos desde joints filtrados
# from src.data.render_skeleton import render_joints_batch
# 
# if Paths.GENERATED_JOINTS_NPZ.exists():
#     data = np.load(str(Paths.GENERATED_JOINTS_NPZ))
#     render_joints_batch(data['joints'][:500], prefix='skeleton')
#     print('✅ Esqueletos renderizados en internal/data/skeleton_images/')

print('Ejecutable vía: python main.py --render --n 500')

In [ ]:
# Entrenamiento del Image GAN
# from src.training.train_image import train_image
# train_image()

print('Ejecutable vía: python main.py --train-image')
print('Tarda varias horas en GPU con 200 épocas.')

In [ ]:
# Visualizar imágenes generadas tras entrenar
from src.models.image_gan import Generator as ImgG

if Paths.IMAGE_CHECKPOINT.exists():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    G_img = ImgG().to(device)
    ckpt = torch.load(str(Paths.IMAGE_CHECKPOINT), map_location=device)
    G_img.load_state_dict(ckpt['G_state_dict'])
    G_img.eval()
    
    with torch.no_grad():
        z = torch.randn(8, 128, device=device)
        fake = G_img(z).cpu()
    
    fake = (fake + 1) / 2
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, img in zip(axes.flat, fake):
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.axis('off')
    plt.suptitle('Muestras generadas por el Image GAN')
    plt.tight_layout(); plt.show()
else:
    print('Aún no hay checkpoint del Image GAN.')

---
## 6. GAN 3D — nubes de puntos (mallas)

Aquí generamos **nubes de 6890 puntos en 3D**, que representan la superficie del cuerpo. 6890 es la cantidad exacta de vértices del modelo SMPL, así podemos comparar directamente con mallas reales.

### ¿Qué es una nube de puntos?

Una nube de puntos es un conjunto de puntos 3D sin conectividad explícita. Es más fácil de aprender que una malla completa (que requeriría predecir también las caras entre puntos), y se puede visualizar directamente.

### Arquitectura

- **Generador**: MLP que toma ruido z y produce `6890 × 3 = 20.670` números, que reconfiguramos a `(6890, 3)`.
- **Discriminador**: usa la arquitectura **PointNet** (Qi et al. 2017), el estándar para redes sobre nubes de puntos:
  - Aplica un MLP a cada punto individualmente (compartiendo pesos).
  - Agrega con **max-pooling global** → un vector de características que resume toda la nube.
  - Clasifica ese vector global.
  
  La clave de PointNet es que es **invariante al orden de los puntos**: da igual cómo estén listados, la red ve lo mismo.

### Datos

- **NOMO3D**: 356 escaneos 3D reales de personas, submuestreados a 6890 puntos.
- **Mallas SMPL sintéticas**: generadas a partir de poses aleatorias pequeñas y betas variados. Esto aumenta el dataset sin violar la semántica del modelo SMPL.

In [ ]:
# Visualizar una nube de puntos real de NOMO3D
from src.data.nomo3d_loader import NOMO3DDataset

try:
    nomo = NOMO3DDataset(gender='all')
    sample = nomo[0].numpy()  # (6890, 3)
    
    fig = plt.figure(figsize=(6, 8))
    ax = fig.add_subplot(111, projection='3d')
    # Submuestreamos para que matplotlib no se atore
    idx = np.random.choice(len(sample), 1500, replace=False)
    ax.scatter(sample[idx, 0], sample[idx, 1], sample[idx, 2], s=1, c='steelblue')
    ax.set_title('Escaneo real de NOMO3D (submuestreado a 1500 pts para visualización)')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'⚠️ {e}')

In [ ]:
# Generar mallas SMPL sintéticas (alimentan al Mesh GAN)
# from src.data.smpl_mesh_generator import generate_and_save_meshes
# generate_and_save_meshes(n_meshes=500)

print('Ejecutable vía: python main.py --gen-meshes --n 500')

In [ ]:
# Entrenamiento del Mesh GAN
# from src.training.train_mesh import train_mesh
# train_mesh()

print('Ejecutable vía: python main.py --train-mesh')

---
## 7. Métricas — ¿cómo medimos si la GAN genera bien?

Evaluar un modelo generativo es **difícil** porque no hay una "respuesta correcta" para cada entrada. Usamos métricas que comparan la **distribución** de los datos generados con la **distribución** de los datos reales.

### 7.1 Tabular: MMD y Bone Length Error

- **MMD (Maximum Mean Discrepancy)**: imagina que proyectas tus datos reales y tus datos generados a un espacio de características usando un kernel (aquí usamos el kernel RBF). Si las dos nubes de puntos quedan superpuestas en ese espacio, MMD ≈ 0. Si están separadas, MMD > 0. **Menor es mejor.**

- **Bone Length Error**: calculamos las longitudes de los huesos (p. ej. antebrazo, fémur) en datos reales y en datos generados. Si las longitudes promedio son parecidas, significa que el generador respeta la anatomía. **Menor es mejor.**

### 7.2 Imagen: FID

- **FID (Fréchet Inception Distance)**: usa una red Inception preentrenada para extraer un vector de características de cada imagen. Luego compara las estadísticas (media y covarianza) de los vectores reales vs generados. **Menor es mejor.** Valores típicos en papers: 10-50 es decente, <10 es bueno, >100 es malo.

### 7.3 3D: Chamfer Distance y F-Score

- **Chamfer Distance**: para cada punto en la nube A, busca el punto más cercano en B, y viceversa. Si todos los puntos encuentran un vecino cercano, CD ≈ 0. **Menor es mejor.**
- **F-Score@τ**: fracción de puntos con un vecino a distancia ≤ τ (aquí τ = 1cm). Es una versión más interpretable del CD. **Mayor es mejor** (máximo 1.0).

In [ ]:
# Evaluación completa — necesita los 3 checkpoints entrenados
results = {}

try:
    from src.evaluation.eval_tabular import evaluate_tabular
    results['tabular'] = evaluate_tabular()
except Exception as e:
    print(f'⚠️ Tabular eval: {e}')

try:
    from src.evaluation.eval_image import evaluate_image
    results['image_fid'] = evaluate_image()
except Exception as e:
    print(f'⚠️ Image eval: {e}')

try:
    from src.evaluation.eval_3d import evaluate_3d
    results['3d'] = evaluate_3d()
except Exception as e:
    print(f'⚠️ 3D eval: {e}')

print('\n=== Resumen ===')
for k, v in results.items():
    print(f'  {k}: {v}')

---
## 8. Conclusiones

### Lo que hemos construido

Un pipeline reproducible que entrena tres GANs distintas (tabular, imagen, 3D) sobre datos reales y sintéticos del cuerpo humano, con:

- **Entrenamiento estable** gracias a WGAN-GP.
- **Filtrado por calidad** usando el propio Discriminador como selector.
- **Evaluación cuantitativa** con métricas estándar (MMD, FID, Chamfer, F-score).
- **Documentación accesible** (este notebook) y un orquestador `main.py` por fases.

### Qué ha funcionado y qué no

*(Esta sección se rellena tras ejecutar los entrenamientos con los resultados reales.)*

### Próximos pasos

- **GAN condicional**: en vez de generar poses aleatorias, condicionar el generador con parámetros antropométricos (altura, peso) o con categorías de movimiento.
- **Decoder joints → SMPL params**: entrenar una red auxiliar que convierta los 72 números del Tabular GAN en parámetros SMPL válidos, lo que permitiría generar mallas coherentes con las poses.
- **Secuencias temporales**: extender el Tabular GAN a un modelo recurrente o transformer que genere trayectorias de movimiento, no solo poses estáticas.
- **Métricas perceptuales**: hacer un "test de Turing" con humanos para evaluar si las poses generadas son indistinguibles de las reales.

---
## Apéndice — cómo ejecutar el pipeline completo

Desde la raíz del proyecto:

```bash
# Ejecutar todo (tarda días en CPU, horas en GPU)
python main.py --all

# O paso a paso:
python main.py --preprocess            # AMASS → joints.npz
python main.py --train-tabular          # Entrena Tabular GAN (~horas)
python main.py --filter --n 1000        # Filtra cuerpos buenos (score > 0.8)
python main.py --gen-meshes --n 500     # Genera mallas SMPL sintéticas
python main.py --render --n 500         # Renderiza esqueletos a PNG
python main.py --train-image            # Entrena Image GAN (~horas)
python main.py --train-mesh             # Entrena Mesh GAN (~horas)
python main.py --eval                   # Calcula MMD, FID, Chamfer, F-score
```